# Applied Machine Learning

## Assignment2 - ShunFai Lee

### Question1:

From chapter 2 and 3 of the text book, each classifiers in this question can be briefly described and compared as follows:
- Perceptron (textbook's version) 
    - Perceptron is the most basic type of machine learning, which employ the idea of using a decision function that perform a binary classification task with a changing weighting on different features and minimizing the overall classification errors.
    - speed
        - very fast to train and predict.
    - strength and robustness
        - very simple in idea, it solves an optimization problem with the classification errors as cost function.
    - Disadvantages
        - training of the model will never converges if the target classes are not perfectly linearly separable.
- SVM
    - The SVM models is like a perceptron but with a different optimization objective, i.e. to maximize a margin, which is defined as the distance between the separating hyperplane (decision boundary) and the training examples that are closest to that hyperplane.
    - speed: 
        - this model is generally fast in prediction, although might involve computing extra metrics. Training, however, is slower than perceptron because it might involve additional calculations of higher dimensional features.
    - strength and robustness
        - this model can be kernelized to solve nonlinear classification problems by combining original features and projecting to higher dimensional space.
    - Disadvantages
        - Need careful decision on hyperparameter C and different gamma value to control overfitting and variance of the model.
- Decision Tree
    - A decision tree model learns a series of questions to break the samples down into categories until at all leave nodes, all remaining samples are in the same class. The learning process is achieved with an iterative process on optimizing an information gain function and the impurity measures.
    - speed: 
        - Moderate for training, because at each nodes, the information gain and impurity measures are calculated repeatedly. Though, prediction is very fast because of the nature of tree traversal.
    - strength and robustness
        - it has good interpretability by visualizing the decision tree model and looking at the splitting criterion at each node.
    - Disadvantages
        - One need to decide on a maximum depth of tree to avoid overfitting.
- Random Forest
    - A random forest can be considered as an ensemble of multiple decision trees that we can average the result from multiple decision trees to get better generalization performance.
    - speed: Slow
    - strength and robustness
        - can average multiple decision trees that individually suffer from high variance to build a more robust model.
        - less susceptible to overfitting.
        - don’t have to worry about choosing good hyperparameter values.
    - Disadvantages
        - The strength is achieved at the expense of an increased computational cost, on both training and prediction stages.



In [56]:
#import all needed libraries
from sklearn import datasets, metrics, svm
from sklearn.model_selection import train_test_split
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns; sns.set_theme(style="ticks", color_codes=True) 
import pandas as pd
from io import StringIO

In [ ]:
#load iris data set
iris = datasets.load_iris()
#manual extract from breastcancer, imdb and bike dataset from kaggle
data = """age,menopause,tumor-size,inv-nodes,node-caps,deg-malig,breast,breast-quad,irradiat,recurrence
44,premeno,21,2,no,2,right,left_up,no,no-recurrence-events"""
breastcancer = pd.read_csv(StringIO(data))
data = """review,sentiment
"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fact that it goes where other shows wouldn't dare. Forget pretty pictures painted for mainstream audiences, forget charm, forget romance...OZ doesn't mess around. The first episode I ever saw struck me as so nasty it was surreal, I couldn't say I was ready for it, but as I watched more, I developed a taste for Oz, and got accustomed to the high levels of graphic violence. Not just violence, but injustice (crooked guards who'll be sold out for a nickel, inmates who'll kill on order and get away with it, well mannered, middle class inmates being turned into prison bitches due to their lack of street skills or prison experience) Watching Oz, you may become comfortable with what is uncomfortable viewing....thats if you can get in touch with your darker side.",positive"""
imdb = pd.read_csv(StringIO(data))
data = """instant,dteday,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
1,01-01-2018,1,0,1,0,6,0,2,14.110847,18.18125,80.5833,10.749882,331,654,985"""
bike = pd.read_csv(StringIO(data))
#load digits
digit = datasets.load_digits()

In [92]:
print(f"example of numerical feature: data set iris: feature: {iris["feature_names"][0]}, entry 0: value: {iris["data"][0,0]}")
print(f"example of nominal feature: data set breast_cancer: feature: {breastcancer.columns[-1]}, entry 0: value: {breastcancer.iloc[0].iloc[-1]}")
print(f"example of date feature: data set Bike Sharing Dataset: feature: {bike.columns[1]}, entry 0: value: {bike[bike.columns[1]].iloc[0]}")
print(f"example of text feature: data set imdb comments: feature: {imdb.columns[0]}, entry 0: value: {imdb.iloc[-1].iloc[0][:20]}...")
print(f"example of dependent variable: data set digits: entry 0: value of label: {digit["target"][0]}")

example of numerical feature: data set iris: feature: sepal length (cm), entry 0: value: 5.1
example of nominal feature: data set breast_cancer: feature: recurrence, entry 0: value: no-recurrence-events
example of date feature: data set Bike Sharing Dataset: feature: dteday, entry 0: value: 01-01-2018
example of text feature: data set imdb comments: feature: review, entry 0: value: One of the other rev...
example of dependent variable: data set digits: entry 0: value of label: 0


### Question2:

- Numerical features
    - Numerical features are features that can be described as discrete(e.g. integers) or continous(e.g. float) values that are ordinal. An example of numerical values is the "sepal length" of iris dataset, e.g. at entry 0, "sepal length" = 5.1cm
- Nominal
    - Nominal feature are simply categorical data and could or could not have order. The subset of that with ordering is ordinal type nominal features. An example of nominal feature is "recurrence" in the breast cancer data set, e.g. entry 0, "recurrence" = no-recurrence-events.
- Date 
    - Date feature is a chronological data that reflect a day, in a month, in a year, and is ordinal. And a way to store and represent it conveniently in a model feature is to take a reference point (e.g. January 1, 1900 in Excel) and stored every date after that as an integer values that represent the number of days between the reference and the date. An example of date feature is "dteday" in Bike Sharing Dataset. e.g. entry 0, "dteday" = 01-01-2018.
- Text 
    - Text feature is a data type that contains series of characters, in human languages. An example of nominal feature is "review" in the imdb review data set, e.g. entry 0, "review" = "No one expects the S...".
- Image 
    - Image feature is some arrays of pixel value in rgb or grayscale that represent an image for purpose of learning visual characteristics for classification. It might not be a complete image as commonly understood but a corner/cropped part of an image that represent some features. An example is Obstacle Detection Dataset from kraggle. Each image is annotated with bounding box of some objects, which are then used as input to train the object detection model. For example, the "IMG_00001.jpg" contains 1 annotation of a car object.
- Dependent variable 
    - It is the class/label or predicated class/label of a data entry, that its value depends on the features. An example is the "target" in digits dataset. e.g. entry 0, "target" = 0.

### Question3:

Based on a Machine Learning article from [Google's crash course](https://developers.google.com/machine-learning/crash-course/classification/accuracy-precision-recall), 
There are several other performance metrics that could help us evaluating a model, besides accuracy, which are Recall, or true positive rate, False positive rate and Precision. But before introducing each of them, a definition is given below for some of the paramemters needed for these three metrics, based on how we could group predicted results:

- True Positive (TP) : A sample correctly predicted for belonging to a class
- True Negative (TN) : A sample correctly predicted for not belonging to a class
- False Positive (FP) : A sample incorrectly predicted for belonging to a class
- False Negative (FN) : A sample incorrectly predicted for not belonging to a class

1. Recall, or true positive rate (TPR)
    - this is the proportion of all actual positives that were classified correctly as positives, or in other word, the detection rate of a class. Mathematically, it is calculated as: $$TPR=\frac{TP}{TP+FN}$$
2. False positive rate (FPR)
    - this is the proportion of negatives that were classified as positives, or in other word, the rate of a false detection/classification. Mathematically, it is calculated as: $$FPR=\frac{FP}{TN+FP}$$
3. Precision
    - this is the proportion of detected positives that were actually real possitives, or in other word, the rate of correct class detections out of the claimed truth. Mathematically, it is calculated as: $$Precision=\frac{TP}{TP+FP}$$

### Question4:

#### Correlation program for Graduate Admission dataset

In [95]:
#load wine data set
admission = pd.read_csv("Admission_Predict.csv")
admission

,Serial No.,GRE Score,TOEFL Score,University Rating,SOP,LOR,CGPA,Research,Chance of Admit
0,1,337,118,4,4.5,4.5,9.65,1,0.92
1,2,324,107,4,4.0,4.5,8.87,1,0.76
2,3,316,104,3,3.0,3.5,8.00,1,0.72
3,4,322,110,3,3.5,2.5,8.67,1,0.80
4,5,314,103,2,2.0,3.0,8.21,0,0.65
...,...,...,...,...,...,...,...,...,...
395,396,324,110,3,3.5,3.5,9.04,1,0.82
396,397,325,107,3,3.0,3.5,9.11,1,0.84
397,398,330,116,4,5.0,4.5,9.45,1,0.91
398,399,312,103,3,3.5,4.0,8.78,0,0.67


In [ ]:
admission_processed = admission.drop(columns='Serial No.')

#features_mean = np.zeros(len(admission.columns))

,GRE Score,TOEFL Score,University Rating,SOP,LOR,CGPA,Research,Chance of Admit
0,337,118,4,4.5,4.5,9.65,1,0.92
1,324,107,4,4.0,4.5,8.87,1,0.76
2,316,104,3,3.0,3.5,8.00,1,0.72
3,322,110,3,3.5,2.5,8.67,1,0.80
4,314,103,2,2.0,3.0,8.21,0,0.65
...,...,...,...,...,...,...,...,...
395,324,110,3,3.5,3.5,9.04,1,0.82
396,325,107,3,3.0,3.5,9.11,1,0.84
397,330,116,4,5.0,4.5,9.45,1,0.91
398,312,103,3,3.5,4.0,8.78,0,0.67
